# Step 3: Teacher Tree Interpretation

The aim is to understand the upper-level split structure, the most frequently used features, and the largest leaf rules with sufficient sample support.


In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "hdb_resale_final_modeling_dataset_2015_2025.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")

PROJECT_ROOT = find_project_root()
DT_V4_DIR = PROJECT_ROOT / "results" / "DT_models"
if str(DT_V4_DIR) not in sys.path:
    sys.path.insert(0, str(DT_V4_DIR))


In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from dt_modelsv4_utils import JSON_PATH

summary = json.loads(JSON_PATH.read_text(encoding="utf-8"))

interpretation = summary["step3_teacher_interpretation"]

display(Markdown("## Tree Overview"))
overview = pd.DataFrame(
    [
        {
            "depth": interpretation["depth"],
            "leaf_count": interpretation["leaf_count"],
            "root_rule": interpretation["root_summary"]["rule"],
        }
    ]
)
display(overview)

split_usage = pd.DataFrame(
    [
        {"feature": feature, "internal_node_count": count}
        for feature, count in interpretation["split_feature_usage"].items()
    ]
).sort_values(["internal_node_count", "feature"], ascending=[False, True])
display(Markdown("## Most-Used Split Features"))
display(split_usage.head(10))

top_leaf_rules = pd.DataFrame(interpretation["top_leaf_rules_by_samples"])
if not top_leaf_rules.empty:
    top_leaf_rules = top_leaf_rules.loc[:, ["samples", "prediction", "path"]].copy()
    top_leaf_rules["prediction"] = top_leaf_rules["prediction"].round(2)
display(Markdown("## Largest Leaf Rules"))
display(top_leaf_rules)

display(Markdown("## Teacher Tree Figure"))
display(Image(filename=summary["artifacts"]["teacher_png_path"]))
